# 🏛️ Capstone Project: Cease & Desist Document Processing System

An intelligent **multi-agent system** built with LangGraph that automates the classification and processing of Cease & Desist documents.

## System Architecture

```
PDF Document
    │
    ▼
Document Loader  →  Classifier  →  [Router]
                                       │
                        ┌──────────────┼──────────────┐
                        ▼              ▼              ▼
                  Database Agent  Archiving Agent  HITL Agent
                        │              │              │ (interrupt)
                        │              │              ▼
                        │              │         Human Review
                        │              │              │
                        └──────────────┴──────────────┘
                                       │
                                       ▼
                                  Audit Agent  →  audit.log
```

| Agent | Responsibility |
|---|---|
| **Document Loader** | Extracts text from PDF using pdfplumber |
| **Classifier** | Uses Claude to classify doc as cease / irrelevant / uncertain |
| **Database Agent** | Writes valid C&D requests to SQLite |
| **Archiving Agent** | Appends irrelevant docs to CSV |
| **HITL Agent** | Interrupts graph for human review of uncertain docs |
| **Audit Agent** | Logs every decision and action to audit.log |

---
## Section 1 — Setup & Dependencies

In [377]:
# Install required packages
%pip install langchain langchain-groq langgraph -q

In [378]:
import json
import sqlite3
import csv
import pdfplumber
from datetime import datetime
from pathlib import Path
from typing import TypedDict, Optional, List, Annotated
import operator

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

print("✅ All imports successful")

✅ All imports successful


---
## Section 2 — Configuration & Initialization

Set API key and define all file paths. The system creates the SQLite database and CSV archive automatically if they don't exist.

In [379]:
# ── API Key ──────────────────────────────────────────────────────────────────
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# LangSmith Tracing
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "Cease-Desist"

# ── File Paths ────────────────────────────────────────────────────────────────
BASE_DIR    = Path(".")
PDF_DIR     = BASE_DIR / "sample_pdfs"
DB_PATH     = BASE_DIR / "cease_desist.db"   # SQLite database
ARCHIVE_CSV = BASE_DIR / "irrelevant_archive.csv"
AUDIT_LOG   = BASE_DIR / "audit.log"

# Create sample_pdfs directory if it doesn't exist
PDF_DIR.mkdir(exist_ok=True)

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

print("✅ Configuration loaded")
print(f"   PDF Directory : {PDF_DIR.resolve()}")
print(f"   Database      : {DB_PATH.resolve()}")
print(f"   Archive CSV   : {ARCHIVE_CSV.resolve()}")
print(f"   Audit Log     : {AUDIT_LOG.resolve()}")

✅ Configuration loaded
   PDF Directory : /content/sample_pdfs
   Database      : /content/cease_desist.db
   Archive CSV   : /content/irrelevant_archive.csv
   Audit Log     : /content/audit.log


In [380]:
# ── Initialize SQLite Database ────────────────────────────────────────────────
def initialize_database():
    """Create the cease_requests table if it doesn't already exist."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS cease_requests (
            id                   INTEGER PRIMARY KEY AUTOINCREMENT,
            document_name        TEXT    NOT NULL,
            received_date        TEXT    NOT NULL,
            sender_name          TEXT,
            sender_address       TEXT,
            date_of_letter       TEXT,
            demands              TEXT,
            classification_reason TEXT,
            decision_source      TEXT,   -- 'ai' or 'human'
            created_at           TEXT    DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()
    print("✅ SQLite database ready")

# ── Initialize CSV Archive ────────────────────────────────────────────────────
def initialize_csv():
    """Create the irrelevant archive CSV with headers if it doesn't exist."""
    if not ARCHIVE_CSV.exists():
        with open(ARCHIVE_CSV, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["document_name", "received_date", "archived_at"])
    print("✅ Archive CSV ready")

initialize_database()
initialize_csv()

✅ SQLite database ready
✅ Archive CSV ready


---
## Section 3 — Shared State

In [381]:
class GraphState(TypedDict):
    # ── Input ──────────────────────────────────────────────────────────────────
    document_path:         str                          # Path to the PDF file

    # ── Document Loader Output ─────────────────────────────────────────────────
    document_name:         str                          # Filename (e.g. letter.pdf)
    received_date:         str                          # Date processing started (YYYY-MM-DD)
    extracted_text:        str                          # Raw text from PDF

    # ── Classifier Output ──────────────────────────────────────────────────────
    classification:        str                          # 'cease' | 'irrelevant' | 'uncertain'
    classification_reason: str                          # Plain-English explanation
    extracted_details:     dict                         # Sender name, address, demands, etc.

    # ── HITL Output ────────────────────────────────────────────────────────────
    human_decision:        str                          # 'cease' | 'irrelevant' (set post-review)
    decision_source:       str                          # 'ai' | 'human'

    # ── Audit Trail (append-only across all agents) ────────────────────────────
    audit_trail: Annotated[List[str], operator.add]     # Each agent appends entries here

    # ── Error Handling ─────────────────────────────────────────────────────────
    error:                 Optional[str]                # Set if any agent fails

print("✅ GraphState defined")

✅ GraphState defined


---
## Section 4 — Tools (Database & File Helpers)

These are pure helper functions (not LangGraph nodes). The agents call them internally to perform side effects.

In [382]:
# ── Database Tool ─────────────────────────────────────────────────────────────
def save_to_database(state: GraphState) -> bool:
    """
    Insert a cease & desist record into the SQLite database.
    Returns True on success, False on failure.
    """
    try:
        details = state.get("extracted_details", {})
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO cease_requests
                (document_name, received_date, sender_name, sender_address,
                 date_of_letter, demands, classification_reason, decision_source)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            state["document_name"],
            state["received_date"],
            details.get("sender_name",    "Unknown"),
            details.get("sender_address", "Unknown"),
            details.get("date_of_letter", "Unknown"),
            details.get("demands",        "Unknown"),
            state.get("classification_reason", ""),
            state.get("decision_source", "ai")
        ))
        conn.commit()
        conn.close()
        return True
    except Exception as e:
        print(f"❌ Database error: {e}")
        return False


# ── Archive Tool ──────────────────────────────────────────────────────────────
def save_to_archive(state: GraphState) -> bool:
    """
    Append an irrelevant document record to the CSV archive.
    Returns True on success, False on failure.
    """
    try:
        with open(ARCHIVE_CSV, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                state["document_name"],
                state["received_date"],
                datetime.now().isoformat()
            ])
        return True
    except Exception as e:
        print(f"❌ Archive error: {e}")
        return False


# ── Database Viewer (helper for demo) ────────────────────────────────────────
def view_database():
    """Print all cease requests stored in the database."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM cease_requests")
    rows = cursor.fetchall()
    conn.close()
    if not rows:
        print("No cease requests in database yet.")
        return
    print(f"{'─'*80}")
    print(f"{'ID':<4} {'Document':<30} {'Sender':<20} {'Date':<12} {'Source':<8}")
    print(f"{'─'*80}")
    for row in rows:
        print(f"{row[0]:<4} {row[1]:<30} {row[3]:<20} {row[2]:<12} {row[8]:<8}")
    print(f"{'─'*80}")

print("✅ Tools defined")

✅ Tools defined


---
## Section 5 — Agent Nodes

Each agent is a plain Python function that receives `state` and returns a **partial state dict** (only the keys it updates). LangGraph merges the returned dict back into the full state automatically.

In [383]:
# ── Node 1: Document Loader ───────────────────────────────────────────────────
def document_loader_node(state: GraphState) -> dict:
    """
    Reads the PDF at document_path using pdfplumber and extracts all text.
    Populates: document_name, received_date, extracted_text.
    """
    path = Path(state["document_path"])
    timestamp = datetime.now().isoformat()

    try:
        with pdfplumber.open(path) as pdf:
            pages_text = [page.extract_text() or "" for page in pdf.pages]
        extracted_text = "\n".join(pages_text).strip()


        log = f"[{timestamp}] DOCUMENT_LOADER: Extracted {len(extracted_text)} chars from '{path.name}'"
        print(f"  📄 {log}")

        return {
            "document_name":  path.name,
            "received_date":  datetime.now().strftime("%Y-%m-%d"),
            "extracted_text": extracted_text,
            "audit_trail":    [log]
        }

    except Exception as e:
        log = f"[{timestamp}] DOCUMENT_LOADER: ERROR reading '{path.name}' — {e}"
        print(f"  ❌ {log}")
        return {
            "document_name":  path.name,
            "received_date":  datetime.now().strftime("%Y-%m-%d"),
            "extracted_text": "",
            "error":          str(e),
            "audit_trail":    [log]
        }

print("✅ document_loader_node defined")

✅ document_loader_node defined


In [384]:
# ── Node 2: Classifier ────────────────────────────────────────────────────────

CLASSIFICATION_PROMPT = """You are a legal document classifier specializing in Cease & Desist requests.

Analyze the document text below and classify it into exactly one category:
  - "cease"      : A valid cease and desist request explicitly asking the company
                   to stop all direct communication or specified activities.
  - "irrelevant" : Not a cease and desist request (e.g., general complaint,
                   inquiry, invoice, unrelated correspondence).
  - "uncertain"  : The intent is ambiguous — could be a cease request but lacks
                   clear language, is incomplete, or is written in a way that
                   requires human judgment.

Also extract these fields if present in the document:
  - sender_name     : Full name of the person sending the document
  - sender_address  : Sender's address
  - date_of_letter  : Date written on the letter itself
  - demands         : What the sender is demanding (stop calls, emails, visits, etc.)
  - recipient_name  : Who the letter is addressed to

Respond ONLY with a valid JSON object. No preamble, no markdown fences.
{{
    "classification": "cease" | "irrelevant" | "uncertain",
    "reason": "Clear explanation of your classification decision",
    "extracted_details": {{
        "sender_name": "...",
        "sender_address": "...",
        "date_of_letter": "...",
        "demands": "...",
        "recipient_name": "..."
    }}
}}

Document text:
---
{text}

---"""


def classifier_node(state: GraphState) -> dict:
    """
    Sends extracted text to Claude and gets back a structured classification.
    Populates: classification, classification_reason, extracted_details, decision_source.
    Falls back to 'uncertain' if text is empty or parsing fails.
    """
    timestamp = datetime.now().isoformat()

    # ── Guard: no text extracted ───────────────────────────────────────────────
    if not state.get("extracted_text", "").strip():
        log = f"[{timestamp}] CLASSIFIER: Empty text — defaulting to 'uncertain'"
        print(f"  ⚠️  {log}")
        return {
            "classification":        "uncertain",
            "classification_reason": "No text could be extracted from the document.",
            "extracted_details":     {},
            "decision_source":       "ai",
            "audit_trail":           [log]
        }

    # ── Call model ────────────────────────────────────────────────────────────

    prompt  = CLASSIFICATION_PROMPT.format(text=state["extracted_text"][:4000])

    response = llm.invoke(prompt)
    import re

    raw = response.content.strip()


    # Robustly extract the first {...} JSON block regardless of surrounding text
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)

    try:
        if not json_match:
            raise json.JSONDecodeError("No JSON object found in response", raw, 0)
        result         = json.loads(json_match.group())
        classification = result["classification"]
        reason         = result["reason"]
        details        = result.get("extracted_details", {})
    except (json.JSONDecodeError, KeyError) as e:
        log = f"[{timestamp}] CLASSIFIER: JSON parse error ({e}) — defaulting to 'uncertain'"
        print(f"  ❌ {log}")
        return {
            "classification":        "uncertain",
            "classification_reason": f"Classification failed due to a parsing error: {e}",
            "extracted_details":     {},
            "decision_source":       "ai",
            "audit_trail":           [log]
        }

    log = (f"[{timestamp}] CLASSIFIER: '{state['document_name']}' → "
           f"'{classification.upper()}' | {reason}")
    print(f"  🤖 {log}")

    return {
        "classification":        classification,
        "classification_reason": reason,
        "extracted_details":     details,
        "decision_source":       "ai",
        "audit_trail":           [log]
    }

print("✅ classifier_node defined")

✅ classifier_node defined


In [385]:
# ── Node 3: Database Agent ────────────────────────────────────────────────────
def database_agent_node(state: GraphState) -> dict:
    """
    Stores a valid Cease & Desist request in the SQLite database.
    """
    timestamp = datetime.now().isoformat()
    success   = save_to_database(state)

    if success:
        log = (f"[{timestamp}] DATABASE_AGENT: Stored '{state['document_name']}' "
               f"in cease_requests table (source: {state.get('decision_source', 'ai')})")
        print(f"  💾 {log}")
    else:
        log = f"[{timestamp}] DATABASE_AGENT: ERROR — Failed to store '{state['document_name']}'"
        print(f"  ❌ {log}")

    return {"audit_trail": [log]}

print("✅ database_agent_node defined")

✅ database_agent_node defined


In [386]:
# ── Node 4: Archiving Agent ───────────────────────────────────────────────────
def archiving_agent_node(state: GraphState) -> dict:
    """
    Appends an irrelevant document record to the CSV archive.
    """
    timestamp = datetime.now().isoformat()
    success   = save_to_archive(state)

    if success:
        log = (f"[{timestamp}] ARCHIVING_AGENT: Archived '{state['document_name']}' "
               f"to irrelevant_archive.csv")
        print(f"  🗂️  {log}")
    else:
        log = f"[{timestamp}] ARCHIVING_AGENT: ERROR — Failed to archive '{state['document_name']}'"
        print(f"  ❌ {log}")

    return {"audit_trail": [log]}

print("✅ archiving_agent_node defined")

✅ archiving_agent_node defined


In [387]:
# ── Node 5: HITL Agent ────────────────────────────────────────────────────────
def hitl_node(state: GraphState) -> dict:
    """
    Pauses the graph using LangGraph's interrupt() for human review.

    Flow:
      1. Builds a human-readable summary of the uncertain document.
      2. Calls interrupt(summary) — the graph PAUSES here and returns control
         to the caller (run_pipeline).
      3. When the human resumes the graph via Command(resume='cease' | 'irrelevant'),
         execution continues from the line after interrupt().
      4. The human's decision is written into state so the router can act on it.
    """
    timestamp = datetime.now().isoformat()
    details   = state.get("extracted_details", {})

    # Build a concise summary to present to the human reviewer
    summary = {
        "document_name":         state["document_name"],
        "received_date":         state["received_date"],
        "ai_classification":     state.get("classification", "uncertain"),
        "ai_reason":             state.get("classification_reason", ""),
        "extracted_details":     details,
        "document_preview":      state.get("extracted_text", "")[:600] + "..."
    }

    pre_log = (f"[{timestamp}] HITL_AGENT: '{state['document_name']}' "
               f"flagged as UNCERTAIN — interrupting for human review")
    print(f"  ⏸️  {pre_log}")

    # ── GRAPH PAUSES HERE ─────────────────────────────────────────────────────
    # The value passed to interrupt() is available to the caller.
    # Execution resumes after this line when Command(resume=...) is invoked.
    human_decision = interrupt(summary)
    # ── GRAPH RESUMES HERE ────────────────────────────────────────────────────

    resume_timestamp = datetime.now().isoformat()
    post_log = (f"[{resume_timestamp}] HITL_AGENT: Human reviewed '{state['document_name']}' "
                f"and decided → '{human_decision.upper()}'")
    print(f"  👤 {post_log}")

    return {
        "human_decision":  human_decision,
        "classification":  human_decision,   # Update classification to human's decision
        "decision_source": "human",
        "audit_trail":     [pre_log, post_log]
    }

print("✅ hitl_node defined")

✅ hitl_node defined


---
## Section 6 — Routing Functions

Conditional edges use these functions to decide which node to visit next. They read from state and return a **string key** that maps to a node name.

In [388]:
def route_after_classification(state: GraphState) -> str:
    """
    Routes after the classifier:
      'cease'      → database_agent
      'irrelevant' → archiving_agent
      'uncertain'  → hitl_agent (interrupt for human review)
    """
    classification = state.get("classification", "uncertain")
    route_map = {
        "cease":      "database_agent",
        "irrelevant": "archiving_agent",
        "uncertain":  "hitl_agent"
    }
    destination = route_map.get(classification, "hitl_agent")
    print(f"  🔀 Router → {destination} (classification: '{classification}')")
    return destination


def route_after_hitl(state: GraphState) -> str:
    """
    Routes after human review:
      'cease'      → database_agent
      'irrelevant' → archiving_agent
    """
    decision = state.get("human_decision", "irrelevant")
    destination = "database_agent" if decision == "cease" else "archiving_agent"
    print(f"  🔀 HITL Router → {destination} (human decision: '{decision}')")
    return destination


print("✅ Routing functions defined")

✅ Routing functions defined


---
## Section 7 — Graph Assembly & Compilation

We wire all nodes and edges together, then compile with `MemorySaver` as the checkpointer. **The checkpointer is required for `interrupt()` / resume to work** — it saves graph state between the pause and the resume so no data is lost.

In [389]:
# ── Build the graph ───────────────────────────────────────────────────────────
builder = StateGraph(GraphState)

# Register all nodes
builder.add_node("document_loader", document_loader_node)
builder.add_node("classifier",      classifier_node)
builder.add_node("database_agent",  database_agent_node)
builder.add_node("archiving_agent", archiving_agent_node)
builder.add_node("hitl_agent",      hitl_node)


# ── Edges ─────────────────────────────────────────────────────────────────────
# Linear start
builder.add_edge(START, "document_loader")
builder.add_edge("document_loader", "classifier")

# Branch after classification
builder.add_conditional_edges(
    "classifier",
    route_after_classification,
    {
        "database_agent":  "database_agent",
        "archiving_agent": "archiving_agent",
        "hitl_agent":      "hitl_agent"
    }
)

# Branch after HITL human review
builder.add_conditional_edges(
    "hitl_agent",
    route_after_hitl,
    {
        "database_agent":  "database_agent",
        "archiving_agent": "archiving_agent"
    }
)

builder.add_edge("database_agent",  END)
builder.add_edge("archiving_agent", END)

# ── Compile with MemorySaver (required for interrupt/resume) ──────────────────
checkpointer = MemorySaver()
graph        = builder.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully")
print()
print("Graph nodes:", list(builder.nodes.keys()))

✅ Graph compiled successfully

Graph nodes: ['document_loader', 'classifier', 'database_agent', 'archiving_agent', 'hitl_agent']


---
## Section 8 — Running the System

### How it works
- `run_pipeline(pdf_path)` processes a single PDF using a unique `thread_id` (the filename stem)
- If a document is classified as **uncertain**, the graph **pauses** and the thread is added to `pending_hitl`
- The **HITL Resume cell** below lets you inspect and decide on pending documents
- Each document is fully isolated — an error in one never affects others

In [390]:
# Tracks threads paused at interrupt (document_name_stem → config)
pending_hitl = {}


def run_pipeline(pdf_path: Path):
    """
    Invoke the graph for a single PDF document.

    Each document gets its own thread_id (filename stem), which means:
      - Fully isolated state and checkpoints
      - Errors are contained to that document
      - HITL pauses only affect that one thread
    """
    thread_id = pdf_path.stem  # e.g. 'cease_letter_001'
    config = {
    "configurable": {"thread_id": thread_id},
    "run_name": f"process:{pdf_path.name}",        # shows as trace title in LangSmith
    "tags": ["cease-desist", "document-processing"],
    "metadata": {
        "document_name": pdf_path.name,
        "thread_id":     thread_id
      }
    }

    print(f"\n{'='*60}")
    print(f"📄 Document : {pdf_path.name}")
    print(f"🧵 Thread   : {thread_id}")
    print(f"{'='*60}")

    result = graph.invoke(
        {"document_path": str(pdf_path), "audit_trail": []},
        config=config
    )

    # Check if graph paused at an interrupt (HITL case)
    current_state = graph.get_state(config)
    if current_state.next:
        # Graph is paused — surface interrupt payload to the reviewer
        pending_hitl[thread_id] = config

        print(f"\n{'⚠️ '*20}")
        print(f"  HUMAN REVIEW REQUIRED")
        print(f"  Thread ID : '{thread_id}'")
        print(f"  Document  : {result.get('document_name', 'unknown')}")
        print(f"  AI Reason : {result.get('classification_reason', '')}")

        # Show the interrupt payload (the summary dict from hitl_node)
        for task in current_state.tasks:
            if hasattr(task, 'interrupts') and task.interrupts:
                payload = task.interrupts[0].value
                print(f"\n  📋 Document Preview:")
                print(f"  {payload.get('document_preview', '')[:400]}")
                print(f"\n  Extracted Details: {json.dumps(payload.get('extracted_details', {}), indent=4)}")

        print(f"\n  👉 Run the HITL Resume cell below with thread_id = '{thread_id}'")
        print(f"{'⚠️ '*20}")
    else:
        classification = result.get("classification", "unknown")
        source         = result.get("decision_source", "ai")
        print(f"\n✅ COMPLETE | {classification.upper()} | Source: {source}")

    return result


print("✅ run_pipeline defined")

✅ run_pipeline defined


In [391]:
# ── Main Loop: Process all PDFs in sample_pdfs/ ───────────────────────────────
# Place your PDF files in the sample_pdfs/ directory before running this cell.

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

if not pdf_files:
    print("⚠️  No PDF files found in sample_pdfs/ directory.")
    print(f"   Please add PDFs to: {PDF_DIR.resolve()}")
else:
    print(f"🔍 Found {len(pdf_files)} PDF(s) to process\n")

    for pdf_path in pdf_files:
        try:
            run_pipeline(pdf_path)
        except Exception as e:
            print(f"\n❌ Unhandled error processing '{pdf_path.name}': {e}")
            print("   Skipping to next document...")

    print(f"\n{'='*60}")
    print(f"📊 Processing complete.")
    print(f"   Documents processed : {len(pdf_files)}")
    print(f"   Pending HITL reviews: {len(pending_hitl)}")
    if pending_hitl:
        print(f"   Threads awaiting review: {list(pending_hitl.keys())}")

🔍 Found 15 PDF(s) to process


📄 Document : 01_copyright_infringement_photography.pdf
🧵 Thread   : 01_copyright_infringement_photography
  📄 [2026-03-24T11:44:09.800555] DOCUMENT_LOADER: Extracted 3152 chars from '01_copyright_infringement_photography.pdf'
  🤖 [2026-03-24T11:44:10.034139] CLASSIFIER: '01_copyright_infringement_photography.pdf' → 'CEASE' | The letter explicitly states a formal demand to immediately cease and desist the unauthorized use of copyrighted photographs, lists specific actions to be taken, and threatens legal action if not complied with, which meets the definition of a valid cease and desist request.
  🔀 Router → database_agent (classification: 'cease')
  💾 [2026-03-24T11:44:12.774185] DATABASE_AGENT: Stored '01_copyright_infringement_photography.pdf' in cease_requests table (source: ai)

✅ COMPLETE | CEASE | Source: ai

📄 Document : 02_trademark_infringement_tech.pdf
🧵 Thread   : 02_trademark_infringement_tech
  📄 [2026-03-24T11:44:12.821971] DOCUMENT_LOADER: 

---
## Section 9 — Human-in-the-Loop (HITL) Resume

If any documents were flagged as **uncertain**, they are now paused and waiting in `pending_hitl`.

**Instructions:**
1. Review the document summary printed above for each pending thread
2. Set `hitl_thread_id` to the thread you want to decide
3. Set `hitl_decision` to `"cease"` or `"irrelevant"`
4. Run the cell — the graph will resume and complete processing

In [392]:
# ── Check pending reviews ─────────────────────────────────────────────────────
if pending_hitl:
    print(f"📋 Documents awaiting human review ({len(pending_hitl)}):")
    for tid in pending_hitl:
        print(f"   • thread_id: '{tid}'")
else:
    print("✅ No documents currently awaiting review.")

📋 Documents awaiting human review (5):
   • thread_id: 'bw_doc_1'
   • thread_id: 'bw_doc_2'
   • thread_id: 'bw_doc_3'
   • thread_id: 'bw_doc_4'
   • thread_id: 'bw_doc_5'


In [398]:
# ── HITL Resume ───────────────────────────────────────────────────────────────
# Edit the two variables below, then run this cell.

hitl_thread_id = "bw_doc_3"   # 📝 e.g. 'uncertain_letter_001'
hitl_decision  = "irrelevant"                     # 📝 'cease' or 'irrelevant'

# ─────────────────────────────────────────────────────────────────────────────
if hitl_thread_id not in pending_hitl:
    print(f"❌ No pending HITL thread found for '{hitl_thread_id}'")
    print(f"   Available threads: {list(pending_hitl.keys())}")

elif hitl_decision not in ("cease", "irrelevant"):
    print(f"❌ Invalid decision '{hitl_decision}'. Must be 'cease' or 'irrelevant'.")

else:
    config = pending_hitl[hitl_thread_id]

    print(f"▶️  Resuming thread '{hitl_thread_id}' with decision: '{hitl_decision}'")
    print()

    # Resume the graph — passes the human's decision back into the interrupted node
    result = graph.invoke(Command(resume=hitl_decision), config=config)

    # Clean up
    del pending_hitl[hitl_thread_id]

    print(f"\n✅ COMPLETE | {result.get('classification', 'unknown').upper()} "
          f"| Source: {result.get('decision_source', 'human')}")

    # Show remaining pending
    if pending_hitl:
        print(f"\n⏳ Still pending: {list(pending_hitl.keys())}")
    else:
        print("\n✅ All documents reviewed.")

▶️  Resuming thread 'bw_doc_3' with decision: 'irrelevant'

  ⏸️  [2026-03-24T11:48:46.794079] HITL_AGENT: 'bw_doc_3.pdf' flagged as UNCERTAIN — interrupting for human review
  👤 [2026-03-24T11:48:46.794222] HITL_AGENT: Human reviewed 'bw_doc_3.pdf' and decided → 'IRRELEVANT'
  🔀 HITL Router → archiving_agent (human decision: 'irrelevant')
  🗂️  [2026-03-24T11:48:46.796325] ARCHIVING_AGENT: Archived 'bw_doc_3.pdf' to irrelevant_archive.csv

✅ COMPLETE | IRRELEVANT | Source: human

⏳ Still pending: ['bw_doc_4', 'bw_doc_5']


---
## Section 10 — Results & Audit Viewer

Inspect the output of the system: database records, the CSV archive, and the full audit log.

In [399]:
# ── View SQLite Database ──────────────────────────────────────────────────────
print("🗄️  CEASE & DESIST DATABASE")
print()
view_database()

🗄️  CEASE & DESIST DATABASE

────────────────────────────────────────────────────────────────────────────────
ID   Document                       Sender               Date         Source  
────────────────────────────────────────────────────────────────────────────────
1    01_copyright_infringement_photography.pdf Priya Nair           2026-03-24   ai      
2    02_trademark_infringement_tech.pdf Jonathan Marks       2026-03-24   ai      
3    03_trade_secret_misappropriation.pdf Angela Morrison, Esq. 2026-03-24   ai      
4    04_defamation_online_review.pdf Dr. Samantha Ellis   2026-03-24   ai      
5    05_patent_infringement_medical_device.pdf Robert Tannenbaum, Esq. 2026-03-24   ai      
6    06_harassment_workplace.pdf    Linda Fujimoto       2026-03-24   ai      
7    07_software_license_violation.pdf Michael Brennan, Esq. 2026-03-24   ai      
8    08_non_compete_violation.pdf   Patricia Weldon, Esq. 2026-03-24   ai      
9    09_copyright_infringement_music.pdf Carlos Rivera  

In [400]:
# ── View Irrelevant Archive CSV ───────────────────────────────────────────────
print("🗂️  IRRELEVANT DOCUMENT ARCHIVE")
print()
if ARCHIVE_CSV.exists():
    with open(ARCHIVE_CSV, "r") as f:
        print(f.read())
else:
    print("Archive file not found.")

🗂️  IRRELEVANT DOCUMENT ARCHIVE

document_name,received_date,archived_at
bw_doc_1.pdf,2026-03-24,2026-03-24T11:47:43.369618
bw_doc_3.pdf,2026-03-24,2026-03-24T11:48:46.796455

